In [4]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import scanpy as sc
import ot
import paste as pst
from scipy.sparse import csr_matrix

import os.path as osp

In [3]:
import ot 
import torch
import os
os.environ['CUDA_VISIBLE_DEVICES']= '0'
# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
import pickle as pkl
import time 
import pandas as pd 
import anndata
from sklearn.metrics import adjusted_rand_score


cuda


Simulation data

In [ ]:
slice_ids =[151673,100,80,60,40,20]

slices=[]
slice1=sc.read_h5ad('/home/data3/yuling/xenium_region2_5599/simData/DLPFC_151673original_spotIndex.h5')
slice1.layers["counts"] = slice1.X.copy()
sc.pp.normalize_total(slice1)
sc.pp.log1p(slice1)
slice1.obs['slice_id'] = 151673
slice1.obs['Z'] = 0
slice1.obs_names_make_unique()
slices.append(slice1)
for index,s in enumerate(slice_ids[1:]):
    slice1 = sc.read_h5ad(osp.join('/home/data3/yuling/xenium_region2_5599/simData/DLPFC_151673_overlap='+str(s)+'__pseudocount_0_spotIndex.h5'))
    slice1.layers["counts"] = slice1.X.copy()
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs['slice_id'] = s
    slice1.obs['Z'] = index+1
    slice1.obs_names_make_unique()
    slices.append(slice1) 

In [4]:
pis = []
dataset = []
time_align = []
fold_path = '/home/data3/yuling/xenium_region2_5599/PASTE/res/simu_data'
for i in range(len(slice_ids)-1):
    start_time = time.time()
    slice1, slice2 = slices[i], slices[i+1]
    pi0_12 = pst.match_spots_using_spatial_heuristic(slice1.obsm['spatial'],slice2.obsm['spatial'],use_ot=True)
    pi12 = pst.pairwise_align(slice1, slice2, G_init=pi0_12, norm=True, backend=ot.backend.TorchBackend(),use_gpu = True)
    end_time = time.time()
    elapsed_time = end_time - start_time 
    time_align.append(elapsed_time)
    pis.append(pi12)
    index = str(slice_ids[i]) + '_' + str(slice_ids[i+1])
    map_path = osp.join(fold_path, index + '_AlignmentPi.pkl')
    dataset.append(index)
    with open(map_path,'wb') as f:
        pkl.dump(pi12, f)
run_times = pd.DataFrame({'alignment': time_align},index = dataset)
run_times.to_csv(osp.join(fold_path,'alignment.csv'))
align_total = sum(time_align)

gpu is available, using gpu.
gpu is available, using gpu.


/home/users/yuling/anaconda3/envs/myenv/lib/python3.8/site-packages/ot/lp/__init__.py:354: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  result_code_string = check_result(result_code)


gpu is available, using gpu.
gpu is available, using gpu.
gpu is available, using gpu.


In [9]:
pis_1 = []
dataset_1 = []
time_align_1 = []
fold_path = '/home/data3/yuling/xenium_region2_5599/PASTE/res/simu_data'
for i in range(1, len(slice_ids)):
    start_time = time.time()
    slice1, slice2 = slices[0], slices[i]
    pi0_12 = pst.match_spots_using_spatial_heuristic(slice1.obsm['spatial'],slice2.obsm['spatial'],use_ot=True)
    pi12 = pst.pairwise_align(slice1, slice2, G_init=pi0_12, norm=True, backend=ot.backend.TorchBackend(),use_gpu = True)
    end_time = time.time()
    elapsed_time = end_time - start_time 
    time_align_1.append(elapsed_time)
    pis_1.append(pi12)
    index = str(slice_ids[0]) + '_' + str(slice_ids[i])
    map_path = osp.join(fold_path, index + '_AlignmentPi.pkl')
    dataset_1.append(index)
    with open(map_path,'wb') as f:
        pkl.dump(pi12, f)
run_times = pd.DataFrame({'alignment': time_align_1},index = dataset_1)
run_times.to_csv(osp.join(fold_path,'alignment_2.csv'))


gpu is available, using gpu.
gpu is available, using gpu.


/home/users/yuling/anaconda3/envs/myenv/lib/python3.8/site-packages/ot/lp/__init__.py:354: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  result_code_string = check_result(result_code)


gpu is available, using gpu.


/home/users/yuling/anaconda3/envs/myenv/lib/python3.8/site-packages/ot/lp/__init__.py:354: UserWarning: numItermax reached before optimality. Try to increase numItermax.
  result_code_string = check_result(result_code)


gpu is available, using gpu.
gpu is available, using gpu.


In [5]:
start_time = time.time()
new_slices, matrix, translations = pst.stack_slices_pairwise(slices, pis, output_params=True, matrix = True)
end_time = time.time()
time_3d = end_time - start_time 
run_times_1 = pd.DataFrame({'alignment': align_total,
                            '3d': time_3d},index=[0])
run_times_1.to_csv(osp.join(fold_path,'overall_runtime_breakdown.csv'))
run_time_total = align_total + time_3d
run_time_total = pd.DataFrame({'overall':run_time_total},index=[0])
run_time_total.to_csv(osp.join(fold_path,'overall_runtime.csv'))

In [6]:
aligned_adata = anndata.concat(new_slices)
aligned_adata.obsm['spatial_3D'] = np.concatenate([aligned_adata.obsm['spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns = ['X', 'Y','Z'],
                index = aligned_adata.obs_names.to_list())
    
df['original_clusters'] = aligned_adata.obs['original_clusters'].values
df['slice'] = aligned_adata.obs['slice_id'].values
df.to_csv(osp.join(fold_path,'3d_coord_simu.csv'))

/home/users/yuling/anaconda3/envs/myenv/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


mHypothalamus data

In [ ]:
slices=[]
slice_ids = ['-0.04', '-0.09',  '-0.14', '-0.19' , '-0.24']
for j,section_ids in enumerate(slice_ids):
    slice1 = load_mHypothalamus(section_id=section_ids)
    slice1.var_names_make_unique()
    slice1.layers["counts"] = slice1.X.copy()
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs['Z']=j
    slice1.obs['slice_id']=section_ids
    slices.append(slice1)

In [ ]:
pis = []
dataset = []
time_align = []
fold_path = '/home/data3/yuling/xenium_region2_5599/PASTE/res/mHy'
for i in range(len(slice_ids)-1):
    start_time = time.time()
    slice1, slice2 = slices[i], slices[i+1]
    pi0_12 = pst.match_spots_using_spatial_heuristic(slice1.obsm['spatial'],slice2.obsm['spatial'],use_ot=True)
    pi12 = pst.pairwise_align(slice1, slice2, G_init=pi0_12, norm=True, backend=ot.backend.TorchBackend(),use_gpu = True)
    end_time = time.time()
    elapsed_time = end_time - start_time 
    time_align.append(elapsed_time)
    pis.append(pi12)
    index = slice_ids[i] + '_' + slice_ids[i+1]
    map_path = osp.join(fold_path, index+'_AlignmentPi.pkl')
    dataset.append(index)
    with open(map_path,'wb') as f:
        pkl.dump(pi12, f)
run_times = pd.DataFrame({'alignment': time_align},index = dataset)
run_times.to_csv(osp.join(fold_path,'alignment.csv'))
align_total = sum(time_align)

In [ ]:
start_time = time.time()
new_slices, matrix, translations = pst.stack_slices_pairwise(slices, pis, output_params=True, matrix = True)
end_time = time.time()
time_3d = end_time - start_time 
run_times_1 = pd.DataFrame({'alignment': align_total,
                            '3d': time_3d},index=[0])
run_times_1.to_csv(osp.join(fold_path,'overall_runtime_breakdown.csv'))
run_time_total = align_total + time_3d
run_time_total = pd.DataFrame({'overall':run_time_total},index=[0])
run_time_total.to_csv(osp.join(fold_path,'overall_runtime.csv'))

In [ ]:
aligned_adata = anndata.concat(new_slices)
aligned_adata.obsm['spatial_3D'] = np.concatenate([aligned_adata.obsm['spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns = ['X', 'Y','Z'],
                index = aligned_adata.obs_names.to_list())
    
df['original_clusters'] = aligned_adata.obs['original_clusters'].values
df['slice'] = aligned_adata.obs['slice_id'].values
df.to_csv(osp.join(fold_path,'3d_coord_mHy.csv'))

DLPFC data

In [ ]:
slice_ids = ['151507', '151508',  '151509', '151510']

slices=[]
for j,section_ids in enumerate(slice_ids):
    slice1 = load_DLPFC(section_id = section_ids)
    slice1.layers["counts"] = slice1.X.copy()
    slice1.obs['Z']=j
    slice1.obs['slice_id']=section_ids
    sc.pp.normalize_total(slice1)
    sc.pp.log1p(slice1)
    slice1.obs_names_make_unique()
    slices.append(slice1)

In [ ]:
pis = []
dataset = []
time_align = []
fold_path = '/home/data3/yuling/xenium_region2_5599/PASTE/res/DLPFC_4/151507_151510'
for i in range(len(slice_ids)-1):
    start_time = time.time()
    slice1, slice2 = slices[i], slices[i+1]
    pi0_12 = pst.match_spots_using_spatial_heuristic(slice1.obsm['spatial'],slice2.obsm['spatial'],use_ot=True)
    pi12 = pst.pairwise_align(slice1, slice2, G_init=pi0_12, norm=True, backend=ot.backend.TorchBackend(),use_gpu = True)
    end_time = time.time()
    elapsed_time = end_time - start_time 
    time_align.append(elapsed_time)
    pis.append(pi12)
    index = str(slice_ids[i]) + '_' + str(slice_ids[i+1])
    map_path = osp.join(fold_path, index + '_AlignmentPi.pkl')
    dataset.append(index)
    with open(map_path,'wb') as f:
        pkl.dump(pi12, f)
run_times = pd.DataFrame({'alignment': time_align},index = dataset)
run_times.to_csv(osp.join(fold_path,'alignment.csv'))
align_total = sum(time_align)

In [ ]:
start_time = time.time()
new_slices, matrix, translations = pst.stack_slices_pairwise(slices, pis, output_params=True, matrix = True)
end_time = time.time()
time_3d = end_time - start_time 
run_times_1 = pd.DataFrame({'alignment': align_total,
                            '3d': time_3d},index=[0])
run_times_1.to_csv(osp.join(fold_path,'overall_runtime_breakdown.csv'))
run_time_total = align_total + time_3d
run_time_total = pd.DataFrame({'overall':run_time_total},index=[0])
run_time_total.to_csv(osp.join(fold_path,'overall_runtime.csv'))


In [ ]:
aligned_adata = anndata.concat(new_slices)
aligned_adata.obsm['spatial_3D'] = np.concatenate([aligned_adata.obsm['spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns = ['X', 'Y','Z'],
                index = aligned_adata.obs_names.to_list())
    
df['original_clusters'] = aligned_adata.obs['original_clusters'].values
df['slice'] = aligned_adata.obs['slice_id'].values
df.to_csv(osp.join(fold_path,'3d_coord_151507.csv'))

Merfish Mouse data

In [ ]:
pis = []
dataset = []
time_align = []
fold_path = '/home/data3/yuling/xenium_region2_5599/PASTE/res/Merfish_mouse'
for i in range(len(slice_ids)-1):
    start_time = time.time()
    slice1, slice2 = slices[i], slices[i+1]
    pi0_12 = pst.match_spots_using_spatial_heuristic(slice1.obsm['spatial'],slice2.obsm['spatial'],use_ot=True)
    pi12 = pst.pairwise_align(slice1, slice2, alpha=0, backend=ot.backend.TorchBackend(),use_gpu = True)
    end_time = time.time()
    elapsed_time = end_time - start_time 
    time_align.append(elapsed_time)
    pis.append(pi12)
    index = slice_ids[i] + '_' + slice_ids[i+1]
    map_path = osp.join(fold_path, index+'_AlignmentPi.pkl')
    dataset.append(index)
    with open(map_path,'wb') as f:
        pkl.dump(pi12, f)
run_times = pd.DataFrame({'alignment': time_align},index = dataset)
run_times.to_csv(osp.join(fold_path,'alignment.csv'))
align_total = sum(time_align)

In [ ]:
start_time = time.time()
##### 3d
new_slices, matrix, translations = pst.stack_slices_pairwise(slices, pis, output_params=True, matrix = True)
end_time = time.time()
time_3d = end_time - start_time 
run_times_1 = pd.DataFrame({'alignment': align_total,
                            '3d': time_3d},index=[0])
run_times_1.to_csv(osp.join(fold_path,'overall_runtime_breakdown.csv'))
run_time_total = align_total + time_3d
run_time_total = pd.DataFrame({'overall':run_time_total},index=[0])
run_time_total.to_csv(osp.join(fold_path,'overall_runtime.csv'))

In [ ]:
aligned_adata = anndata.concat(new_slices)
aligned_adata.obsm['spatial_3D'] = np.concatenate([aligned_adata.obsm['spatial'],np.array(aligned_adata.obs['Z'].values)[:,None]],axis=1)
df = pd.DataFrame(aligned_adata.obsm['spatial_3D'],
                columns = ['X', 'Y','Z'],
                index = aligned_adata.obs_names.to_list())
    
df['original_clusters'] = aligned_adata.obs['original_clusters'].values
df['slice'] = aligned_adata.obs['slice_id'].values
df.to_csv(osp.join(fold_path,'3d_coord_merfish.csv'))